###1. Setup

In [0]:
%sql 
create catalog if not exists dbx;
create schema if not exists dbx.scenarios;

###2. ROW_NUMBER

In [0]:
%sql

Select employee_id, department_key, salary, row_number() over(partition by department_key order by salary) as rn from dbx.hr.dim_employee;

No Tie, No Gaps

###3. RANK

In [0]:
%sql
select employee_id, department_key, salary, rank() over(partition by department_key order by salary) as rnk from dbx.hr.dim_employee;

Allowed Ties, Allowed Gaps

###4. DENSE_RANK

In [0]:
%sql
select employee_id, department_key, salary, dense_rank() over(partition by department_key order by salary) as dn from dbx.hr.dim_employee;

Allowed Ties, No Gaps

###5. NTILE

In [0]:
%sql
select employee_id, department_key, salary, NTILE(2) over(partition by department_key order by salary) as buckets from dbx.hr.dim_employee;

In [0]:
%sql
select employee_id, department_key, salary, NTILE(3) over(partition by department_key order by salary) as buckets from dbx.hr.dim_employee;

Splits rows into n buckets as evenly as possible.

###5. Employees earning more than their managers

In [0]:
%sql
CREATE TABLE dbx.scenarios.employee (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    salary INT,
    managerId INT
);

In [0]:
%sql
INSERT INTO dbx.scenarios.employee (id, name, salary, managerId) VALUES
(1, 'Joe',   70000, 3),
(2, 'Henry', 80000, 4),
(3, 'Sam',   60000, NULL),
(4, 'Max',   90000, NULL);

In [0]:
%sql
select e.name, e.salary from dbx.scenarios.employee e 
join dbx.scenarios.employee m
on e.managerId = m.id
where e.salary>m.salary;

###6. Managers with atleast 5 direct report

In [0]:
%sql
-- Create table
CREATE TABLE dbx.scenarios.employee_manager (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    managerId INT
);


In [0]:
%sql
-- Insert sample data
INSERT INTO dbx.scenarios.employee_manager (id, name, department, managerId) VALUES
(101, 'John',  'A', NULL),
(102, 'Dan',   'A', 101),
(103, 'James', 'A', 101),
(104, 'Amy',   'A', 101),
(105, 'Anne',  'A', 101),
(106, 'Ron',   'B', 101);

In [0]:
%sql
select e.managerId, m.name, count(*) as manager from dbx.scenarios.employee_manager e 
join dbx.scenarios.employee_manager m
on e.managerId = m.id
group by 1,2
having count(*)>4


###7. Monthly sales aggregation with rows-to-columns transformation.

In [0]:
%sql
CREATE TABLE dbx.scenarios.salesdata (
    OrderID INT PRIMARY KEY,
    OrderDate DATE,
    SalesAmount DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.salesdata (OrderID, OrderDate, SalesAmount) VALUES
(1,  '2023-01-15', 1000.00),
(2,  '2023-02-10', 1500.00),
(3,  '2023-03-05', 2000.00),
(4,  '2023-04-18', 1800.00),
(5,  '2023-05-22', 2200.00),
(6,  '2023-06-30', 2500.00),
(7,  '2023-07-12', 2700.00),
(8,  '2023-08-09', 3000.00),
(9,  '2023-09-14', 3200.00),
(10, '2023-10-03', 3500.00),
(11, '2023-11-19', 4000.00),
(12, '2023-12-25', 4500.00),
(13, '2024-01-08', 1100.00),
(14, '2024-02-16', 1600.00),
(15, '2024-03-21', 2100.00),
(16, '2024-04-11', 1900.00),
(17, '2024-05-27', 2300.00),
(18, '2024-06-05', 2600.00),
(19, '2024-07-17', 2800.00),
(20, '2024-08-29', 3100.00),
(21, '2024-09-02', 3300.00),
(22, '2024-10-15', 3600.00),
(23, '2024-11-23', 4100.00),
(24, '2024-12-31', 4700.00);

In [0]:
%sql
SELECT
    YEAR(OrderDate) AS SalesYear,

    SUM(CASE WHEN MONTH(OrderDate) = 1  THEN SalesAmount ELSE 0 END) AS Jan,
    SUM(CASE WHEN MONTH(OrderDate) = 2  THEN SalesAmount ELSE 0 END) AS Feb,
    SUM(CASE WHEN MONTH(OrderDate) = 3  THEN SalesAmount ELSE 0 END) AS Mar,
    SUM(CASE WHEN MONTH(OrderDate) = 4  THEN SalesAmount ELSE 0 END) AS Apr,
    SUM(CASE WHEN MONTH(OrderDate) = 5  THEN SalesAmount ELSE 0 END) AS May,
    SUM(CASE WHEN MONTH(OrderDate) = 6  THEN SalesAmount ELSE 0 END) AS Jun,
    SUM(CASE WHEN MONTH(OrderDate) = 7  THEN SalesAmount ELSE 0 END) AS Jul,
    SUM(CASE WHEN MONTH(OrderDate) = 8  THEN SalesAmount ELSE 0 END) AS Aug,
    SUM(CASE WHEN MONTH(OrderDate) = 9  THEN SalesAmount ELSE 0 END) AS Sep,
    SUM(CASE WHEN MONTH(OrderDate) = 10 THEN SalesAmount ELSE 0 END) AS Oct,
    SUM(CASE WHEN MONTH(OrderDate) = 11 THEN SalesAmount ELSE 0 END) AS Nov,
    SUM(CASE WHEN MONTH(OrderDate) = 12 THEN SalesAmount ELSE 0 END) AS Dec

FROM dbx.scenarios.salesdata
GROUP BY YEAR(OrderDate)
ORDER BY SalesYear;

In [0]:
%sql
SELECT *
FROM (
    SELECT
        YEAR(OrderDate) AS SalesYear,
        DATE_FORMAT(OrderDate, 'MMMM') AS SalesMonth,
        SalesAmount
    FROM dbx.scenarios.salesdata
) src
PIVOT (
    SUM(SalesAmount)
    FOR SalesMonth IN (
        'January','February','March','April','May','June',
        'July','August','September','October','November','December'
    )
)
ORDER BY SalesYear;

###8. Running total for different genders

In [0]:
%sql
CREATE TABLE dbx.scenarios.scores (
    player_name VARCHAR(100),
    gender VARCHAR(1),
    day DATE,
    score_points INT,
    PRIMARY KEY (gender, day, player_name)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.scores (player_name, gender, day, score_points) VALUES
('Aron',     'F', '2020-01-01', 17),
('Alice',    'F', '2020-01-07', 23),
('Bajrang',  'M', '2020-01-07', 7),
('Khali',    'M', '2019-12-25', 11),
('Slaman',   'M', '2019-12-30', 13),
('Joe',      'M', '2019-12-31', 3),
('Jose',     'M', '2019-12-18', 2),
('Priya',    'F', '2019-12-31', 23),
('Priyanka', 'F', '2019-12-30', 17);

In [0]:
%sql
SELECT
    gender,
    day,
    SUM(score_points) OVER (
        PARTITION BY gender
        ORDER BY day
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS total
FROM dbx.scenarios.scores
ORDER BY gender, day;

In [0]:
%sql
SELECT
    gender,
    day,
    SUM(score_points) OVER (
        PARTITION BY gender
        ORDER BY day
        --ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS total
FROM dbx.scenarios.scores
ORDER BY gender, day;

###9. Longest Winning streak

In [0]:
%sql
CREATE TABLE dbx.scenarios.matches (
    player_id INT,
    match_day DATE,
    result STRING,
    PRIMARY KEY (player_id, match_day)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.matches (player_id, match_day, result) VALUES
(1, '2022-01-17', 'Win'),
(1, '2022-01-18', 'Win'),
(1, '2022-01-25', 'Win'),
(1, '2022-01-31', 'Draw'),
(1, '2022-02-08', 'Win'),
(2, '2022-02-06', 'Lose'),
(2, '2022-02-08', 'Lose'),
(3, '2022-03-30', 'Win');

In [0]:
%sql
WITH seq AS (
    SELECT 
        player_id,
        match_day,
        result,
        ROW_NUMBER() OVER (
            PARTITION BY player_id 
            ORDER BY match_day
        ) AS match_seq,
        ROW_NUMBER() OVER (
            PARTITION BY player_id, result 
            ORDER BY match_day
        ) AS result_seq
    FROM dbx.scenarios.matches
),

diff AS (
    SELECT 
        player_id,
        match_day,
        match_seq - result_seq AS win_streak_group
    FROM seq
    WHERE result = 'Win'
),

streak AS (
    SELECT 
        player_id,
        win_streak_group,
        COUNT(*) AS win_streak_cnt
    FROM diff
    GROUP BY player_id, win_streak_group
)

SELECT 
    m.player_id,
    COALESCE(MAX(s.win_streak_cnt), 0) AS max_win_streak
FROM dbx.scenarios.matches m
LEFT JOIN streak s 
    ON m.player_id = s.player_id
GROUP BY m.player_id;

###10. Active user retention by Month

In [0]:
%sql
CREATE TABLE dbx.scenarios.user_actions (
    user_id INT,
    event_id INT,
    event_type STRING,
    event_date TIMESTAMP
);

In [0]:
%sql
INSERT INTO dbx.scenarios.user_actions (user_id, event_id, event_type, event_date) VALUES
(445, 7765, 'sign-in', '2022-05-31 12:00:00'),
(742, 6458, 'sign-in', '2022-06-03 12:00:00'),
(445, 3634, 'like',    '2022-06-05 12:00:00'),
(742, 1374, 'comment', '2022-06-05 12:00:00'),
(648, 3124, 'like',    '2022-06-18 12:00:00');

In [0]:
%sql
WITH monthly_active AS (
    SELECT 
        user_id,
        YEAR(event_date)  AS year,
        MONTH(event_date) AS month
    FROM dbx.scenarios.user_actions
    GROUP BY 
        user_id,
        YEAR(event_date),
        MONTH(event_date)
)

SELECT 
    a.year,
    a.month                        AS current_month,
    COUNT(DISTINCT a.user_id)      AS retained_users
FROM monthly_active a
INNER JOIN monthly_active b
    ON  a.user_id = b.user_id
    AND (
        (a.month = b.month + 1 AND a.year = b.year)          -- same year, next month
        OR
        (a.month = 1 AND b.month = 12 AND a.year = b.year + 1) -- Jan of next year after Dec
    )
GROUP BY a.year, a.month
ORDER BY a.year, a.month;

###11. Year on Year growth rate

In [0]:
%sql
CREATE TABLE dbx.scenarios.user_transactions (
    transaction_id     INT,
    product_id         INT,
    spend              DECIMAL(10,2),
    transaction_date   TIMESTAMP
);

In [0]:
%sql
-- INSERT DATA (Example Input)

INSERT INTO dbx.scenarios.user_transactions
(transaction_id, product_id, spend, transaction_date)
VALUES
(1341, 123424, 1500.60, '2019-12-31 12:00:00'),
(1423, 123424, 1000.20, '2020-12-31 12:00:00'),
(1623, 123424, 1246.44, '2021-12-31 12:00:00'),
(1322, 123424, 2145.32, '2022-12-31 12:00:00');

In [0]:
%sql 
WITH yearly_spend AS (
    SELECT 
        YEAR(transaction_date)    AS year,
        product_id,
        SUM(spend)                AS curr_year_spend
    FROM dbx.scenarios.user_transactions
    GROUP BY 
        YEAR(transaction_date),
        product_id
)

SELECT 
    year,
    product_id,
    curr_year_spend,
    LAG(curr_year_spend) OVER (
        PARTITION BY product_id 
        ORDER BY year
    )                             AS prev_year_spend,
    ROUND(
        (curr_year_spend - LAG(curr_year_spend) OVER (
            PARTITION BY product_id ORDER BY year)
        ) 
        / LAG(curr_year_spend) OVER (
            PARTITION BY product_id ORDER BY year)
        * 100
    , 2)                          AS yoy_rate
FROM yearly_spend
ORDER BY product_id, year;

###12. Consecutive for 3 or more filling years

In [0]:
%sql
CREATE TABLE dbx.scenarios.filed_taxes (
    filing_id   INT,
    user_id     VARCHAR(50),
    filing_date TIMESTAMP,
    product     VARCHAR(100)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.filed_taxes VALUES
(1, '1', '2019-04-14', 'TurboTax Desktop 2019'),
(2, '1', '2020-04-15', 'TurboTax Deluxe'),
(3, '1', '2021-04-15', 'TurboTax Online'),
(4, '2', '2020-04-07', 'TurboTax Online'),
(5, '2', '2021-04-10', 'TurboTax Online'),
(6, '3', '2020-04-07', 'TurboTax Online'),
(7, '3', '2021-04-15', 'TurboTax Online'),
(8, '3', '2022-03-11', 'QuickBooks Desktop Pro'),
(9, '4', '2022-04-15', 'QuickBooks Online');

In [0]:
%sql
WITH turbotax_years AS (
    SELECT 
        user_id,
        YEAR(filing_date) AS filing_year
    FROM dbx.scenarios.filed_taxes
    WHERE product LIKE '%TurboTax%'
),

consecutive_check AS (
    SELECT 
        user_id,
        filing_year,
        LAG(filing_year, 1) OVER (PARTITION BY user_id ORDER BY filing_year) AS prev_year_1,
        LAG(filing_year, 2) OVER (PARTITION BY user_id ORDER BY filing_year) AS prev_year_2
    FROM turbotax_years
)

SELECT DISTINCT user_id
FROM consecutive_check
WHERE filing_year - prev_year_1 = 1
  AND prev_year_1 - prev_year_2 = 1
ORDER BY user_id ASC;

###13. Gaps & Islands - Find all ISLANDS (Consecutive Streaks)

In [0]:
%sql
CREATE TABLE dbx.scenarios.logins (
    login_id   INT,
    user_id    INT,
    login_date DATE
);

In [0]:
%sql
INSERT INTO dbx.scenarios.logins VALUES
(1, 101, '2023-01-01'),
(2, 101, '2023-01-02'),
(3, 101, '2023-01-03'),
(4, 101, '2023-01-06'),
(5, 101, '2023-01-07'),
(6, 101, '2023-01-10');

In [0]:
%sql 
WITH ranked AS (
    SELECT 
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY login_date
        )                                         AS row_num
    FROM dbx.scenarios.logins
),

island_groups AS (
    SELECT 
        user_id,
        login_date,
        -- Databricks SQL syntax: use DATEADD or direct subtraction
        DATEADD(DAY, -row_num, login_date)        AS grp
    FROM ranked
)

SELECT 
    user_id,
    MIN(login_date)  AS island_start,
    MAX(login_date)  AS island_end,
    COUNT(*)         AS consecutive_days
FROM island_groups
GROUP BY user_id, grp
ORDER BY island_start;

###14. Gaps & Islands - Find all GAPS (Missing Date Ranges)

In [0]:
%sql
WITH ranked AS (
    SELECT 
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY login_date
        )                                          AS row_num
    FROM dbx.scenarios.logins
),

island_groups AS (
    SELECT 
        user_id,
        login_date,
        DATEADD(DAY, -row_num, login_date)         AS grp
    FROM ranked
),

islands AS (
    SELECT 
        user_id,
        MIN(login_date)                            AS island_start,
        MAX(login_date)                            AS island_end
    FROM island_groups
    GROUP BY user_id, grp
),

gaps AS (
    SELECT 
        user_id,
        DATEADD(DAY, 1, island_end)                AS gap_start,
        DATEADD(DAY, -1, LEAD(island_start) OVER (
            PARTITION BY user_id 
            ORDER BY island_start)
        )                                          AS gap_end
    FROM islands
)

SELECT 
    user_id,
    gap_start,
    gap_end
FROM gaps
WHERE gap_start <= gap_end   -- removes NULLs and invalid gaps
ORDER BY user_id, gap_start;

###15. Gaps & Islands - Longest Consecutive Streak per User

In [0]:
%sql
WITH ranked AS (
    SELECT 
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY login_date
        )                                  AS row_num
    FROM dbx.scenarios.logins
),

island_groups AS (
    SELECT 
        user_id,
        login_date,
        DATEADD(DAY, -row_num, login_date) AS grp   -- Databricks syntax
    FROM ranked
),

streaks AS (
    SELECT 
        user_id,
        COUNT(*)                           AS streak_length
    FROM island_groups
    GROUP BY user_id, grp
)

SELECT 
    user_id,
    MAX(streak_length)                     AS longest_streak
FROM streaks
GROUP BY user_id
ORDER BY longest_streak DESC;

### 16. Find a sale person id who have made sales for 3 or more consecutive days

In [0]:
%sql
CREATE TABLE dbx.scenarios.salesbyperson (
    sale_id         INT,
    salesperson_id  INT,
    sale_date       DATE
);

In [0]:
%sql
-- INSERT SAMPLE DATA

INSERT INTO dbx.scenarios.salesbyperson (sale_id, salesperson_id, sale_date) VALUES
-- Salesperson 101 → has 3 consecutive days (qualifies)
(1, 101, '2023-01-01'),
(2, 101, '2023-01-02'),
(3, 101, '2023-01-03'),
(4, 101, '2023-01-06'),

-- Salesperson 102 → only 2 consecutive days (does NOT qualify)
(5, 102, '2023-02-01'),
(6, 102, '2023-02-02'),
(7, 102, '2023-02-05'),

-- Salesperson 103 → 4 consecutive days (qualifies)
(8, 103, '2023-03-10'),
(9, 103, '2023-03-11'),
(10,103, '2023-03-12'),
(11,103, '2023-03-13'),

-- Salesperson 104 → non-consecutive sales (does NOT qualify)
(12,104, '2023-04-01'),
(13,104, '2023-04-03'),
(14,104, '2023-04-05');

In [0]:
%sql 
WITH ranked_sales AS (
    SELECT 
        salesperson_id,
        sale_date,
        ROW_NUMBER() OVER (
            PARTITION BY salesperson_id 
            ORDER BY sale_date
        )                                      AS row_num
    FROM dbx.scenarios.salesbyperson
    GROUP BY salesperson_id, sale_date         -- one record per day per salesperson
),

grouping_logic AS (
    SELECT 
        salesperson_id,
        sale_date,
        DATEADD(DAY, -row_num, sale_date)      AS grp   -- ✅ Databricks syntax
    FROM ranked_sales
),

consecutive_counts AS (
    SELECT 
        salesperson_id,
        grp,
        MIN(sale_date)                         AS streak_start,
        MAX(sale_date)                         AS streak_end,
        COUNT(*)                               AS consecutive_days
    FROM grouping_logic
    GROUP BY salesperson_id, grp
)

SELECT DISTINCT salesperson_id
FROM consecutive_counts
WHERE consecutive_days >= 3
ORDER BY salesperson_id;

###17. Recursive CTE to flatten out a ragged parent child hierarchy.

In [0]:
%sql
CREATE TABLE dbx.scenarios.parent_child (
    EmployeeID INT,
    EmployeeName VARCHAR(100),
    ManagerID INT
);

In [0]:
%sql
INSERT INTO dbx.scenarios.parent_child (EmployeeID, EmployeeName, ManagerID) VALUES
(1, 'Anna', NULL),   -- CEO
(2, 'John', 1),     -- Reports to Anna
(3, 'Maria', 1),    -- Reports to Anna
(4, 'James', 2),    -- Reports to John
(5, 'Jane', 2),     -- Reports to John
(6, 'Steve', 3);    -- Reports to Maria

In [0]:
%sql 
WITH RECURSIVE EmployeeHierarchy AS (
    -- Anchor member: top-level records (e.g. CEO with no manager)
    SELECT
        EmployeeID,
        EmployeeName,
        ManagerID,
        CAST(EmployeeName AS STRING) AS HierarchyPath,
        1 AS Level
    FROM
        dbx.scenarios.parent_child
    WHERE
        ManagerID IS NULL

    UNION ALL

    -- Recursive member: next level down
    SELECT
        e.EmployeeID,
        e.EmployeeName,
        e.ManagerID,
        CAST(CONCAT(eh.HierarchyPath, ' -> ', e.EmployeeName) AS STRING),
        eh.Level + 1 AS Level
    FROM
        dbx.scenarios.parent_child e
    INNER JOIN
        EmployeeHierarchy eh ON e.ManagerID = eh.EmployeeID
)
SELECT
    *
FROM
    EmployeeHierarchy
ORDER BY
    HierarchyPath;



### 18. De-duping with CTEs

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.scenarios.employee_records (
    EmployeeID    INT,
    EmployeeName  STRING,
    Department    STRING,
    Salary        DECIMAL(10, 2),
    LoadDate      TIMESTAMP
);

In [0]:
%sql
INSERT INTO dbx.scenarios.employee_records VALUES
    (1, 'Alice',   'Engineering', 95000.00, '2024-01-01 10:00:00'),
    (1, 'Alice',   'Engineering', 95000.00, '2024-01-02 11:00:00'), -- duplicate
    (2, 'Bob',     'Marketing',   72000.00, '2024-01-01 09:00:00'),
    (2, 'Bob',     'Marketing',   72000.00, '2024-01-03 14:00:00'), -- duplicate, later load
    (3, 'Charlie', 'Finance',     88000.00, '2024-01-01 08:00:00'),
    (4, 'Diana',   'Engineering', 101000.00,'2024-01-01 07:00:00');

In [0]:
%sql
WITH ranked_records AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY EmployeeID
            ORDER BY LoadDate DESC   -- change to ASC to keep earliest
        ) AS rn
    FROM
        dbx.scenarios.employee_records
),
deduped AS (
    SELECT * EXCEPT (rn)
    FROM ranked_records
    WHERE rn = 1
)
SELECT *
FROM deduped
ORDER BY EmployeeID;

###19. Cohort retention over time

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.scenarios.user_activity (
    user_id INT,
    activity_date DATE
) USING DELTA;


In [0]:
%sql
INSERT INTO dbx.scenarios.user_activity VALUES
-- Cohort: Jan 2024
(1, '2024-01-15'), (1, '2024-02-10'), (1, '2024-03-05'),
(2, '2024-01-20'), (2, '2024-02-15'),
(3, '2024-01-25'),
-- Cohort: Feb 2024
(4, '2024-02-05'), (4, '2024-03-12'),
(5, '2024-02-10');


In [0]:
%sql 
WITH cohorts AS (
    -- Identify the first activity month for each user
    SELECT 
        user_id, 
        TRUNC(MIN(activity_date), 'MM') AS cohort_month
    FROM dbx.scenarios.user_activity
    GROUP BY user_id
),
retention_intervals AS (
    -- Calculate months elapsed since the first activity
    SELECT 
        c.user_id,
        c.cohort_month,
        TRUNC(a.activity_date, 'MM') AS activity_month,
        DATEDIFF(month, c.cohort_month, TRUNC(a.activity_date, 'MM')) AS month_number
    FROM dbx.scenarios.user_activity a
    JOIN cohorts c ON a.user_id = c.user_id
),
cohort_size AS (
    -- Count total unique users per cohort
    SELECT cohort_month, COUNT(DISTINCT user_id) AS total_users
    FROM cohorts
    GROUP BY cohort_month
)
-- Final aggregation to calculate retention rates
SELECT 
    r.cohort_month,
    s.total_users,
    r.month_number,
    COUNT(DISTINCT r.user_id) AS active_users,
    ROUND(COUNT(DISTINCT r.user_id) * 100.0 / s.total_users, 2) AS retention_rate
FROM retention_intervals r
JOIN cohort_size s ON r.cohort_month = s.cohort_month
GROUP BY 1, 2, 3
ORDER BY 1, 3;


###20. FIFO inventory analysis

In [0]:
%sql
CREATE OR REPLACE TABLE dbx.scenarios.inventory_transactions (
    txn_id INT,
    item_id STRING,
    txn_type STRING, -- 'IN' for Purchase, 'OUT' for Sale
    txn_date DATE,
    qty INT,
    unit_cost DECIMAL(10, 2)
) USING DELTA;


In [0]:
%sql
INSERT INTO dbx.scenarios.inventory_transactions VALUES
(1, 'Apple', 'IN',  '2024-01-01', 100, 10.00), -- Layer 1
(2, 'Apple', 'IN',  '2024-01-05', 50,  12.00), -- Layer 2
(3, 'Apple', 'OUT', '2024-01-10', 120, NULL),  -- Sale: Should use 100 from Layer 1, 20 from Layer 2
(4, 'Apple', 'IN',  '2024-01-15', 30,  15.00); -- Layer 3

In [0]:
%sql 
WITH totals AS (
    -- Calculate total units sold per item
    SELECT item_id, SUM(qty) as total_sold
    FROM dbx.scenarios.inventory_transactions
    WHERE txn_type = 'OUT'
    GROUP BY item_id
),
purchase_layers AS (
    -- Create running totals for "IN" transactions to see the "stack"
    SELECT 
        item_id,
        txn_date,
        qty,
        unit_cost,
        SUM(qty) OVER (PARTITION BY item_id ORDER BY txn_date, txn_id) as cumulative_qty
    FROM dbx.scenarios.inventory_transactions
    WHERE txn_type = 'IN'
)
SELECT 
    p.item_id,
    p.txn_date AS purchase_date,
    p.unit_cost,
    -- Calculate how much of this specific purchase layer remains
    CASE 
        WHEN p.cumulative_qty <= COALESCE(t.total_sold, 0) THEN 0
        WHEN p.cumulative_qty - p.qty < COALESCE(t.total_sold, 0) 
            THEN p.cumulative_qty - t.total_sold
        ELSE p.qty
    END AS remaining_qty,
    -- Value of the remaining stock in this layer
    (CASE 
        WHEN p.cumulative_qty <= COALESCE(t.total_sold, 0) THEN 0
        WHEN p.cumulative_qty - p.qty < COALESCE(t.total_sold, 0) 
            THEN p.cumulative_qty - t.total_sold
        ELSE p.qty
    END) * p.unit_cost AS remaining_value
FROM purchase_layers p
LEFT JOIN totals t ON p.item_id = t.item_id
ORDER BY p.item_id, p.txn_date;


###21. Server Utilization time

In [0]:
%sql
CREATE OR REPLACE TABLE dbx.scenarios.server_utilization (
    server_id INT,
    session_status STRING, -- 'start' or 'stop'
    status_time TIMESTAMP
) USING DELTA;


In [0]:
%sql
INSERT INTO dbx.scenarios.server_utilization VALUES
(1, 'start', '2024-03-01 08:00:00'), (1, 'stop',  '2024-03-02 10:00:00'),
(1, 'start', '2024-03-05 12:00:00'), (1, 'stop',  '2024-03-05 15:00:00'),
(2, 'start', '2024-03-01 10:00:00'), (2, 'stop',  '2024-03-01 22:00:00');

In [0]:
%sql
WITH session_pairs AS (
    -- Pair each 'start' with its subsequent 'stop'
    SELECT 
        server_id,
        session_status,
        status_time AS start_time,
        LEAD(status_time) OVER (PARTITION BY server_id ORDER BY status_time) AS stop_time
    FROM dbx.scenarios.server_utilization
),
durations AS (
    -- Calculate duration only for the 'start' rows
    SELECT 
        server_id,
        UNIX_TIMESTAMP(stop_time) - UNIX_TIMESTAMP(start_time) AS duration_seconds
    FROM session_pairs
    WHERE session_status = 'start'
      AND stop_time IS NOT NULL
)
-- Aggregate total time and convert to full days
SELECT 
    FLOOR(SUM(duration_seconds) / 86400) AS total_utilization_days
FROM durations;


###22. Knights tour

In [0]:
%sql
-- Step 1: Create a table for the 8 valid L-shaped knight moves
CREATE OR REPLACE TABLE dbx.scenarios.knight_move_deltas (
    dx INT,
    dy INT
) USING DELTA;


In [0]:
%sql
-- Step 2: Insert the 8 possible move combinations
INSERT INTO dbx.scenarios.knight_move_deltas VALUES 
(1, 2), (1, -2), (-1, 2), (-1, -2), 
(2, 1), (2, -1), (-2, 1), (-2, -1);


In [0]:
%sql 
WITH RECURSIVE knight_tour AS (
    -- Base Case: Starting position at (1,1)
    SELECT 
        1 AS x, 
        1 AS y, 
        CAST('(1,1)' AS STRING) AS visited_path, 
        1 AS step_count
    
    UNION ALL
    
    -- Recursive Step: Find a valid next move
    SELECT 
        kt.x + md.dx AS next_x, 
        kt.y + md.dy AS next_y,
        CONCAT(kt.visited_path, ' -> (', kt.x + md.dx, ',', kt.y + md.dy, ')') AS next_path,
        kt.step_count + 1 AS next_step
    FROM knight_tour kt
    CROSS JOIN dbx.scenarios.knight_move_deltas  md
    WHERE 
        -- Boundary check for a 5x5 board
        kt.x + md.dx BETWEEN 1 AND 5
        AND kt.y + md.dy BETWEEN 1 AND 5
        -- Ensure the square hasn't been visited before
        AND kt.visited_path NOT LIKE CONCAT('%(', kt.x + md.dx, ',', kt.y + md.dy, ')%')
        -- Stop when the tour is complete (25 squares for 5x5)
        AND kt.step_count < 25
)
-- Select the first completed tour found
SELECT visited_path 
FROM knight_tour 
WHERE step_count = 25 
LIMIT 1;


###23. 100 prisoners recursion

In [0]:
%sql
CREATE OR REPLACE TABLE dbx.scenarios.boxes AS
SELECT 
    id AS box_id,
    row_number() OVER (ORDER BY rand()) AS slip_number
FROM range(1, 101);


In [0]:
%sql 
WITH RECURSIVE prisoner_search AS (
    -- Base Case: Every prisoner starts with the box matching their ID
    SELECT 
        p.id AS prisoner_id,
        b.slip_number AS current_slip,
        1 AS boxes_opened,
        (p.id = b.slip_number) AS found_own_number
    FROM range(1, 101) AS p(id)
    JOIN dbx.scenarios.boxes b ON p.id = b.box_id

    UNION ALL

    -- Recursive Step: Open the box labeled with the previous slip's number
    SELECT 
        ps.prisoner_id,
        b.slip_number,
        ps.boxes_opened + 1,
        (ps.prisoner_id = b.slip_number)
    FROM prisoner_search ps
    JOIN dbx.scenarios.boxes b ON ps.current_slip = b.box_id
    WHERE ps.found_own_number = false 
      AND ps.boxes_opened < 50
)
-- Final Summary: Did everyone survive?
SELECT 
    COUNT(CASE WHEN found_own_number THEN 1 END) = 100 AS all_survived,
    COUNT(CASE WHEN found_own_number THEN 1 END) AS successful_prisoners
FROM prisoner_search
WHERE found_own_number = true OR boxes_opened = 50;


###24. Markov chain recursion

In [0]:
%sql
CREATE OR REPLACE TABLE dbx.scenarios.transition_matrix (
    source_state STRING,
    target_state STRING,
    probability DOUBLE
) USING DELTA;


In [0]:
%sql
INSERT INTO dbx.scenarios.transition_matrix VALUES
('Sunny', 'Sunny', 0.8), ('Sunny', 'Rainy', 0.2),
('Rainy', 'Sunny', 0.4), ('Rainy', 'Rainy', 0.6);

In [0]:
%sql
WITH RECURSIVE markov_paths AS (
    -- 1. Base Case: Starting state
    SELECT 
        'Sunny' AS state, 
        cast(1.0 as double) AS current_prob, 
        0 AS step
    
    UNION ALL
    
    -- 2. Recursive Step: No Aggregates allowed here!
    -- We just calculate individual path probabilities
    SELECT 
        tm.target_state,
        (mp.current_prob * tm.probability) AS next_prob,
        mp.step + 1
    FROM markov_paths mp
    JOIN dbx.scenarios.transition_matrix tm ON mp.state = tm.source_state
    WHERE mp.step < 6 -- Keep this low; paths grow exponentially (2^N)
)
-- 3. Final Step: Aggregate outside the recursion
SELECT 
    state, 
    SUM(current_prob) AS total_prob
FROM markov_paths
WHERE step = 6
GROUP BY state;
